# Module 2: Talking to Your Deep Agent from a UI

> ~30 min. Builds on the research agent from Module 1.

In Module 1 you called `agent.invoke()` right inside the notebook. A real app works differently: the agent runs on a **LangGraph server**, and the UI talks to it over HTTP through the **LangGraph SDK**.

This notebook plays the part of the UI. Each section is one thing a user does in a chat app, the SDK call behind it, and a printout of what that panel would show:

| What the user does | SDK call |
|---|---|
| Opens the app and sees their conversations | `threads.search()` |
| Starts a new conversation | `threads.create()` |
| Sends a message and watches the reply type out | `runs.stream(stream_mode="messages-tuple")` |
| Watches the agent's plan and files update live | `runs.stream(stream_mode="updates")` |
| Sees what the subagents are doing | `runs.stream(..., stream_subgraphs=True)` |
| Approves a file write | `runs.stream(..., command={"resume": ...})` |
| Refreshes the page mid-run, then hits Stop | `runs.join_stream()`, `runs.cancel()` |
| Reopens an old conversation | `threads.get_state()` |

Three words come up everywhere:

- **Assistant**: an agent the server can run. Here, the deep agent from `agent.py`.
- **Thread**: one conversation. The server saves its messages, todos, and files.
- **Run**: one turn of the agent on a thread, started by a user message.

## Setup: start the agent server

`langgraph.json` at the workshop root tells the server what to serve: two graphs, both defined in `agent.py`.

- `research_agent`: the Module 1 research agent (web research tool, research subagent, todos, files).
- `research_agent_approval`: the same agent, but it pauses before every file write so a person can approve it.

You'd normally start the server in a terminal with `langgraph dev`. To keep everything in one place, the next cell starts it in the background and writes its logs to `langgraph_dev.log`. Run the last cell of the notebook to stop it.

In [ ]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

import httpx

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.ui import (
    file_text, render_approval_card, render_files, render_plan, render_sidebar,
    render_transcript, render_update, text_of,
)

SERVER_URL = "http://127.0.0.1:2024"

def server_is_up():
    try:
        return httpx.get(f"{SERVER_URL}/ok", timeout=1).status_code == 200
    except httpx.HTTPError:
        return False

server = None
if server_is_up():
    print("Reusing the agent server already running at", SERVER_URL)
else:
    # Use the `langgraph` CLI installed next to this kernel's Python.
    langgraph_cli = shutil.which("langgraph", path=str(Path(sys.executable).parent)) or "langgraph"
    log_file = open(project_root / "langgraph_dev.log", "w")
    server = subprocess.Popen(
        [langgraph_cli, "dev", "--no-browser", "--no-reload", "--port", "2024"],
        cwd=project_root,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(90):
        if server_is_up():
            break
        if server.poll() is not None:
            raise RuntimeError("The server exited. See langgraph_dev.log for the error.")
        time.sleep(1)
    else:
        raise RuntimeError("The server didn't start within 90 seconds. See langgraph_dev.log.")
    print("Agent server running at", SERVER_URL)

## 1. Connect and find the agent

`get_sync_client()` returns a client for the server. A Python web backend would use this one; `get_client()` is the async version with the same methods.

The server lists one **assistant** per graph in `langgraph.json`. You can refer to an assistant by its graph name, so the UI only needs to know `"research_agent"`.

In [ ]:
from langgraph_sdk import get_sync_client

client = get_sync_client(url=SERVER_URL)

for assistant in client.assistants.search():
    print(f"{assistant['graph_id']:<26} {assistant['assistant_id']}")

## 2. The sidebar: a user's conversations

Each conversation is a **thread**. A UI tags threads with `metadata` (here a `user_id` and a `title`) so it can list just one user's conversations with `threads.search()`.

Each thread also has a `status`, which a sidebar can show as a badge:

| Status | Meaning |
|---|---|
| `idle` | Nothing running. Ready for the next message. |
| `busy` | A run is in progress. |
| `interrupted` | Paused waiting for a person (section 6), or stopped by the user (section 7). |
| `error` | The last run failed. |

The first lines delete Ada's threads from earlier runs of this notebook, so the sidebar starts empty. That's also how a UI's "delete conversation" button works.

In [ ]:
USER = "ada"

# Start clean: remove Ada's conversations from earlier runs of this notebook.
for old in client.threads.search(metadata={"user_id": USER}, limit=100):
    client.threads.delete(old["thread_id"])

# The user clicks "New chat" twice.
chat = client.threads.create(metadata={"user_id": USER, "title": "Quick question"})
research = client.threads.create(metadata={"user_id": USER, "title": "Deep Agents research"})

# A different user's thread. It shouldn't show up in Ada's sidebar.
client.threads.create(metadata={"user_id": "grace", "title": "Grace's chat"})

render_sidebar(USER, client.threads.search(metadata={"user_id": USER}))

## 3. Send a message and stream the reply

`runs.stream()` starts a run and streams it back as it happens. `stream_mode` picks what you receive. A chat bubble wants **tokens**, so use `"messages-tuple"`: each event carries a message chunk plus metadata about where it came from (which graph node, which agent).

The first event is always `metadata`. It holds the `run_id`, which the UI keeps so the Stop button can cancel this run later.

If the run fails (a bad API key, for example), the stream sends an `error` event. A UI would show it in the chat; here every stream loop raises it.

In [ ]:
stream = client.runs.stream(
    chat["thread_id"],
    "research_agent",
    input={"messages": [{"role": "user", "content": "In two sentences, what is a deep agent?"}]},
    stream_mode="messages-tuple",
)

print("🤖 ", end="")
for chunk in stream:
    if chunk.event == "error":
        raise RuntimeError(f"The run failed: {chunk.data}")
    if chunk.event == "metadata":
        run_id = chunk.data["run_id"]
    elif chunk.event == "messages":
        message, metadata = chunk.data
        if message["type"] == "AIMessageChunk":
            print(text_of(message), end="", flush=True)

print(f"\n\n(run_id {run_id})")

## 4. Live plan and files panels

Deep agents keep two things in their state that a UI can show beside the chat:

- `todos`: the plan the agent writes with `write_todos`.
- `files`: the agent's virtual filesystem.

With `stream_mode="updates"` you get one event each time a step of the agent finishes, holding just what that step changed. When a step changes `todos` or `files`, redraw that panel. Tool calls and results become an activity feed.

This run asks for research and a written summary, so expect a plan, a `task()` call to the research subagent, and a new file.

In [ ]:
stream = client.runs.stream(
    research["thread_id"],
    "research_agent",
    input={"messages": [{"role": "user", "content": (
        "Make a short todo plan first. Then research what LangChain Deep Agents is "
        "and write a 3-bullet summary to /summary.md."
    )}]},
    stream_mode="updates",
)

for chunk in stream:
    if chunk.event == "error":
        raise RuntimeError(f"The run failed: {chunk.data}")
    if chunk.event == "updates":
        for step_name, update in chunk.data.items():
            render_update(update)

Notice the activity feed only shows `task()` going out and a result coming back. The research itself happened inside the subagent, and this stream didn't include it.

## 5. See what the subagents are doing

A deep agent's subagents run as **subgraphs**. Pass `stream_subgraphs=True` and the stream includes their steps too. The event name then tells you where each step came from:

- `updates`: the main agent.
- `updates|tools:<id>`: a subagent the main agent started with `task()`.

A UI can use that to nest subagent activity under the `task()` call that started it.

In [ ]:
stream = client.runs.stream(
    research["thread_id"],
    "research_agent",
    input={"messages": [{"role": "user", "content": (
        "Now delegate one quick search on the latest LangGraph release and tell me the version in one sentence."
    )}]},
    stream_mode="updates",
    stream_subgraphs=True,
)

for chunk in stream:
    if chunk.event == "error":
        raise RuntimeError(f"The run failed: {chunk.data}")
    if chunk.event.startswith("updates"):
        namespace = chunk.event.partition("|")[2]
        prefix = "    [subagent] " if namespace else "[main] "
        for step_name, update in chunk.data.items():
            render_update(update, prefix=prefix)

## 6. Approve a tool call

The `research_agent_approval` graph was built with `interrupt_on={"write_file": True, "edit_file": True}`, the same human-in-the-loop setup as Module 1. When the agent tries to write a file, the run pauses and the stream sends an `__interrupt__` event describing the tool call. The thread's status becomes `interrupted`, so a sidebar can show that it's waiting.

The UI draws an approval card from that event.

In [ ]:
approval_thread = client.threads.create(metadata={"user_id": USER, "title": "Write a file"})

stream = client.runs.stream(
    approval_thread["thread_id"],
    "research_agent_approval",
    input={"messages": [{"role": "user", "content": "Write a file /hello.md that says 'Hello from the UI'."}]},
    stream_mode="updates",
)

for chunk in stream:
    if chunk.event == "error":
        raise RuntimeError(f"The run failed: {chunk.data}")
    if chunk.event == "updates" and "__interrupt__" in chunk.data:
        for interrupt in chunk.data["__interrupt__"]:
            render_approval_card(interrupt)

print("\nThread status:", client.threads.get(approval_thread["thread_id"])["status"])

The user clicks a button. The UI resumes the paused run by starting a new stream on the same thread with `command={"resume": ...}` and no `input`. The resume value holds one decision per pending tool call.

Try switching `decision` to the reject line and re-running the cell above, then this one.

In [ ]:
decision = {"type": "approve"}
# decision = {"type": "reject", "message": "Not now, thanks."}

stream = client.runs.stream(
    approval_thread["thread_id"],
    "research_agent_approval",
    command={"resume": {"decisions": [decision]}},
    stream_mode="updates",
)

for chunk in stream:
    if chunk.event == "error":
        raise RuntimeError(f"The run failed: {chunk.data}")
    if chunk.event == "updates":
        for step_name, update in chunk.data.items():
            render_update(update)

print("\nThread status:", client.threads.get(approval_thread["thread_id"])["status"])

## 7. Refresh the page mid-run, then hit Stop

Streaming ties a run to one HTTP connection. Real UIs also need to handle a page refresh while the agent is still working. The pattern:

1. `runs.create()` starts the run **in the background** and returns right away with its `run_id`.
2. `runs.join_stream()` attaches to that run's stream, from any client, at any time. That's what the UI does after a refresh.
3. `runs.cancel()` is the Stop button.

Below, the UI attaches, watches for a few seconds, and then the user hits Stop.

In [ ]:
long_thread = client.threads.create(metadata={"user_id": USER, "title": "Long task"})

run = client.runs.create(
    long_thread["thread_id"],
    "research_agent",
    input={"messages": [{"role": "user", "content": (
        "Plan and research three topics one by one: LangGraph, LangChain, and Deep Agents. "
        "Write a report for each to /reports/."
    )}]},
    stream_mode="updates",
)
print("Started run", run["run_id"], "| thread status:", client.threads.get(long_thread["thread_id"])["status"])

# The page refreshes. The UI reattaches to the run that's still going.
started = time.time()
for chunk in client.runs.join_stream(long_thread["thread_id"], run["run_id"]):
    if chunk.event == "error":
        raise RuntimeError(f"The run failed: {chunk.data}")
    if chunk.event == "updates":
        for step_name, update in chunk.data.items():
            render_update(update)
    if time.time() - started > 8:
        break

# The user hits Stop.
client.runs.cancel(long_thread["thread_id"], run["run_id"], wait=True)
print("\nRun status:   ", client.runs.get(long_thread["thread_id"], run["run_id"])["status"])
print("Thread status:", client.threads.get(long_thread["thread_id"])["status"])

The run shows `interrupted` because the user stopped it. The thread isn't stuck, though: the user can send their next message right away.

This follow-up uses `runs.wait()`, which runs the agent and returns its final state without streaming. That suits short replies, or a backend that doesn't stream to its client.

In [ ]:
final_state = client.runs.wait(
    long_thread["thread_id"],
    "research_agent",
    input={"messages": [{"role": "user", "content": "Never mind. In one sentence, what is LangGraph?"}]},
)
print("🤖", text_of(final_state["messages"][-1]))
print("Thread status:", client.threads.get(long_thread["thread_id"])["status"])

## 8. Reopen a conversation

When the user clicks an old conversation, the UI has nothing in memory. `threads.get_state()` returns everything the server saved for the thread: `values` holds the full agent state, so the UI can rebuild the chat, the plan, and the files in one call.

In [ ]:
state = client.threads.get_state(research["thread_id"])
values = state["values"]

render_transcript(values["messages"])
print()
if values.get("todos"):
    render_plan(values["todos"])
if values.get("files"):
    render_files(values["files"])

summary = values.get("files", {}).get("/summary.md")
if summary:
    print("\n/summary.md\n" + file_text(summary))

## Recap

| UI feature | SDK call |
|---|---|
| List conversations | `threads.search(metadata=...)` |
| New or deleted conversation | `threads.create()`, `threads.delete()` |
| Streaming chat bubble | `runs.stream(stream_mode="messages-tuple")` |
| Plan and file panels | `runs.stream(stream_mode="updates")`, reading `todos` and `files` |
| Subagent activity | `stream_subgraphs=True` |
| Approval card | `__interrupt__` event, then `command={"resume": {"decisions": [...]}}` |
| Survive a refresh, Stop button | `runs.create()`, `runs.join_stream()`, `runs.cancel()` |
| Reopen a conversation | `threads.get_state()` |

**Building the real UI.** A web frontend would usually be React, using `useStream` from `@langchain/langgraph-sdk/react`. It wraps these same calls: it streams runs, keeps the thread's messages and state up to date, and exposes interrupts.

**Further reading**

- [Streaming API](https://docs.langchain.com/langsmith/streaming): every `stream_mode`.
- [Threads](https://docs.langchain.com/langsmith/use-threads): creating, searching, and copying threads.
- [Event streaming API](https://docs.langchain.com/langsmith/event-streaming) (Beta): `client.threads.stream()`, a newer API with typed streams for messages, tool calls, and subgraphs.

## Shut down the server

In [ ]:
import os
import signal

if server is not None:
    if hasattr(os, "killpg"):
        os.killpg(server.pid, signal.SIGTERM)  # stop the server and the worker processes it started
    else:
        server.terminate()
    server.wait(timeout=15)
    print("Agent server stopped.")
else:
    print("This notebook didn't start the server, so it's left running.")